# 🚀 ENHANCED NOM OCR PIPELINE (KẾT HỢP PHƯƠNG ÁN 1 & PHƯƠNG ÁN 2)
### Huấn luyện SE-ResNet trên 54.094 nhãn sạch mở rộng & Tăng cường nét bút lông cổ

Notebook này chạy trên **Kaggle GPU (Tesla T4 hoặc P100)**.

**Hai cải tiến đột phá kết hợp trong bài toán:**
1. **Phương án 1 (Two-Stage OCR)**: Huấn luyện mạng nơ-ron chuyên sâu `EnhancedNomOCRNet` (kiến trúc Residual Blocks kết hợp Squeeze-and-Excitation Attention) trên toàn bộ **54.094 nhãn sạch mở rộng** (51.922 GOLD + 2.172 nhãn vừa được giải cứu ở Vòng 1).
2. **Phương án 2 (Stroke Enhancement & Ink-Denoising)**: Tích hợp module xử lý hình thái học mô phỏng đặc tính ngọn bút lông chép tay thế kỷ 19: cân bằng sáng nền giấy cổ, mô phỏng mực đẫm loang (Dilation) và mực khô khát bút (Erosion), biến dạng chữ thảo.

**Cài đặt:** Ở panel bên phải -> **Settings -> Accelerator -> GPU T4 x2 (hoặc GPU P100)**.

In [ ]:
!nvidia-smi

### 1. Tự động định vị tệp dữ liệu (`crops.npz`, `labels_expanded.csv`, `QuocNgu_SinoNom.csv`)

In [ ]:
import os, sys, glob
from pathlib import Path

crops_files = glob.glob("/kaggle/input/**/crops.npz", recursive=True) + glob.glob("**/crops.npz", recursive=True)
labels_files = glob.glob("/kaggle/input/**/labels_expanded.csv", recursive=True) + glob.glob("**/labels_expanded.csv", recursive=True)
dict_files = glob.glob("/kaggle/input/**/QuocNgu_SinoNom.csv", recursive=True) + glob.glob("**/QuocNgu_SinoNom.csv", recursive=True)
script_files = glob.glob("/kaggle/input/**/train_enhanced_ocr.py", recursive=True) + glob.glob("**/train_enhanced_ocr.py", recursive=True)

assert len(crops_files) > 0, "Không tìm thấy crops.npz! Hãy kiểm tra đã Add Dataset vào notebook chưa."
assert len(labels_files) > 0, "Không tìm thấy labels_expanded.csv!"
assert len(dict_files) > 0, "Không tìm thấy QuocNgu_SinoNom.csv!"

crops_path = crops_files[0]
labels_path = labels_files[0]
dict_path = dict_files[0]
script_path = script_files[0] if script_files else "train_enhanced_ocr.py"

print(f"✓ crops.npz:           {crops_path} ({os.path.getsize(crops_path) / (1024*1024):.2f} MB)")
print(f"✓ labels_expanded.csv: {labels_path} ({os.path.getsize(labels_path) / (1024*1024):.2f} MB)")
print(f"✓ QuocNgu_SinoNom.csv: {dict_path} ({os.path.getsize(dict_path) / (1024*1024):.2f} MB)")
print(f"✓ train script:        {script_path}")

### 2. Chạy Huấn Luyện Enhanced Nom OCR Net (20 Epochs)
Thời gian ước tính trên GPU Tesla T4: **~9–12 phút** (20 epochs kèm Stroke Augmentation & Mixed Precision FP16).

In [ ]:
!python "{script_path}" \
    --crops "{crops_path}" \
    --labels "{labels_path}" \
    --dict "{dict_path}" \
    --epochs 20 \
    --batch-size 128 \
    --lr 0.001 \
    --out-dir /kaggle/working/output

### 3. Đánh giá & Xem nhanh kết quả huấn luyện

In [ ]:
import json
import pandas as pd

metrics_path = "/kaggle/working/output/training_metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)
    df_m = pd.DataFrame(m)
    print("--- TIẾN TRÌNH HUẤN LUYỆN 20 EPOCHS ---")
    display(df_m.tail(10))

res_path = "/kaggle/working/output/enhanced_pseudo_labels.csv"
if os.path.exists(res_path):
    df_res = pd.read_csv(res_path)
    print("\n--- THỐNG KÊ QUYẾT ĐỊNH GIẢI CỨU ---")
    print(df_res["decision"].value_counts())
    
    print("\n--- MẪU 10 Ô ĐƯỢC GIẢI CỨU ĐỘ TIN CẬY CAO ---")
    high = df_res[df_res["decision"] == "RESCUE_V3_HIGH"].head(10)
    display(high[["book", "page", "column", "syllable", "ocr_char_old", "predicted_nom", "unicode", "prob"]])

### 4. Đóng gói kết quả để tải về (`enhanced_results.zip`)

In [ ]:
!cd /kaggle/working/output && zip -r /kaggle/working/enhanced_results.zip .
print("✓ ĐÃ ĐÓNG GÓI HOÀN TẤT: /kaggle/working/enhanced_results.zip")
print("👉 Hãy tải file 'enhanced_results.zip' ở panel bên phải về và đặt vào:")
print("   lab/enhanced_self_training_v3/enhanced_results.zip")